# **Overview**
The goal of this research is to examine how social determinants of health—specifically race, education level, and age—impact maternal mortality rates among Black and White women in the United States.

The research question being answered is: To what extent does maternal mortality vary for Black and White women across different ages, education levels, and U.S. Census regions?


To answer this, the following analyses will be conducted

1.   Evaluate the variability in maternal death rates across U.S. Census regions.
2.   Assess the impact of age on maternal death rates
3.   Examine the correlation between education level and maternal death rates.

The dataset used for this study was obtained from CDC WONDER and includes pregnancy-related deaths among Black and White women from 2018 to 2023. A comparative analysis between the two groups will be performed.

# **Business Case**

Maternal mortality rates in the United States are the highest among high-income nations [Commonwealth Fund, 2024](https://www.commonwealthfund.org/publications/issue-briefs/2024/jun/insights-us-maternal-mortality-crisis-international-comparison). According to the Centers for Disease Control and Prevention [(CDC)](https://www.cdc.gov/maternal-mortality/php/data-research/index.html), over 80% of pregnancy-related deaths are considered preventable. In 2020, cardiovascular conditions were one of the leading causes of maternal death.

Heart disease remains the leading cause of death for women in the U.S., affecting over 60 million women nationwide. Obesity, which exacerbates cardiovascular risk, also increases pregnancy-related complications including diabetes and hypertensive disorders [(Mayo Clinic)](https://www.mayoclinic.org/healthy-lifestyle/pregnancy-week-by-week/in-depth/pregnancy-and-obesity/art-20044409). Regional obesity rates will be briefly considered to contextualize underlying health disparities.

The aim of this research is to explore how U.S. Census region, age, and education level influence pregnancy-related death rates, with the goal of identifying potential target demographics for public health interventions designed to lower maternal mortality rates.


# **Understanding Data**
The dataset used in this research was obtained from CDC WONDER. It includes maternal death information filtered by age ranges 15–44, education levels from no high school diploma or GED to master's degree, all four U.S. Census regions, and Black and White women from 2018–2023. CDC WONDER can be accessed at: https://wonder.cdc.gov/.

**Note:** Some data in this dataset has been suppressed to protect patient confidentiality. As a result, the findings are specific to the available dataset and may not represent the full population.

### **Import Pandas Library and Dataset**



In [ ]:
#import pandas as pd
import pandas as pd

In [ ]:
#Upload the CSV file and read it into a DataFrame
from google.colab import files
uploaded = files.upload()

#read cvs 'maternal_deaths' to DataFrame
df = pd.read_csv('pregnancy_deaths.csv')

###**Data Quality Check**

Before making any changes view the dataset.

In [ ]:
##DataFrame pre-changes

df.info()
df.describe()
df.head()


**Observation:**
1. The dataset includes columns with repeated information. For example, "Single Race 6 Code" and "Single Race 6" contain the same data; however, the former is specific to CDC database grouping.

2. The following columns are contain repeated information and can be deleted: "Single Race 6 Code", "Five-Year Age Groups Code", "Education Code", & "Census Region Code".

3. Columns "Population" and "Crude Rate" contain "Not Applicable" cell which are due to the suppression of data to protect patient confidentiality. These columns can be deleted.  

4. The "Notes" column contains "NaN" values which are not neccessary for analysis and thus can be deleted.

5. The data type for "Deaths" and "% of Total Deaths" are objects and will have to be converted to a numerical data type for analysis.



**Let's look at the summary of the dataset for missing values and duplicate values.**

---




In [ ]:
# Get a summary of the DataFrame
def data_quality_report(df):
    print("Missing Values:\n", df.isnull().sum(), "\n")
    print("Duplicate Rows:", df.duplicated().sum(), "\n")

# Run the function
data_quality_report(df)

**Observation:**


1.   The dataset has five duplicate rows which need to be examined further before analysis.
2.  There is a moderate amount of null values in this dataset which will need to be accounted for.



**View the entire dataframe to see where null values are located.**

In [ ]:
# prompt: view entire dataframe

import pandas as pd
pd.set_option("display.max_rows", None, "display.max_columns", None)
print(df)
pd.reset_option("display.max_rows", "display.max_columns")


**Observation:**
1. The dataset consists of 169 rows, with data populated up to row 111. The remaining rows (112 to 169) contain only null values across all columns and contribute no meaningful information. Therefore, these rows can be safely removed.

2. The "Notes" column contains 112 null ("NaN") values through row 111. Starting at row 112, it includes data related to data suppression measures for patient confidentiality, which has already been documented.

3. It is safe to delete all null values found in the dataset due to their redundancy.


**View Duplicate Rows to See Data Validity**

In [ ]:
# view 5 duplicate rows
duplicates = df[df.duplicated()]
print(duplicates)

**Observation**
1. The five duplicate rows contain null values (empty cells) in each column, making the data unusable. These rows can be safely deleted without negatively impacting the integrity of the remaining dataset.

### **Data Cleaning**

**Remove columns with null or duplicate data**

The repeated columns in this dataset were identified as redundant for analysis and thus are removed.

In [ ]:
# Drop multiple columns at once
columns_to_drop = [
    'Notes',
    'Single Race 6 Code',
    'Five-Year Age Groups Code',
    'Education Code',
    'Census Region Code',
    'Population',
    'Crude Rate'
]

df.drop(columns=columns_to_drop, inplace=True)

# Print first few rows after dropping columns
print("DataFrame after dropping selected columns:\n", df.head())



**Drop rows with null values**

The rows with null values were identified as redundant for analysis and thus are removed.

In [ ]:
#drop rows with only NaN
df = df.dropna(how='all')

# View entire dataset
df.head(111)

**Final check for updated missing values**

In [ ]:
#Check for duplicates
# Get a summary of the DataFrame
def data_quality_report(df):
    print("Missing Values:\n", df.isnull().sum(), "\n")
    print("Duplicate Rows:", df.duplicated().sum(), "\n")
# Run the function
data_quality_report(df)

This is the dataset post changes without null values and redundant columns:

In [ ]:
#dataframe post changes
df.head(112)

### **Standardize Formatting**

The dataset will be standardized for uniformity.

**Removed "years" from the end of age groups in the "Five-Year Age Groups" column.**

In [ ]:
#Remove years from age groups
df['Five-Year Age Groups'] = df['Five-Year Age Groups'].str.replace(r'\s*[Yy]ears?', '', regex=True).str.strip()

#view changes
df.head(20)

**Grammar mistakes in the Education level column were corrected.**

In [ ]:
#Correct grammar and remove degree minor specifications

# Replace ?s with 's
df['Education'] = df['Education'].str.replace(r'\?s', "'s", regex=True)

# Remove anything in brackets
df['Education'] = df['Education'].str.replace(r'\(.*?\)', '', regex=True)

#view new changes
df.head(20)

**The Cenus Region column was changed to only have the name the region in each cell without Census coding.**



In [ ]:
# Change census region to only give region names
df['Census Region'] = df['Census Region'].str.split(':').str[-1].str.strip()

df.head()


**Changing Data Types**

Deaths column was changed to an integer from an object.

In [ ]:
#Change deaths to integer
df['Deaths'] = df['Deaths'].fillna(0).astype(int)
df.head()

The "% of Total Deaths" column was changed from object to a float and the "%" sign dropped from the values.

In [ ]:
#Cast as a string
df['% of Total Deaths'] = df['% of Total Deaths'].astype(str)

#String % from coloumn
df['% of Total Deaths'] = df['% of Total Deaths'].str.replace('%', '', regex=False).str.strip()

#Change % of Total Deaths to float
df['% of Total Deaths'] = df['% of Total Deaths'].astype(float)

#view changes
df.head()

**The headers are renamed for simplicity.**

In [ ]:
#Rename "Single Race 6" to "Race"
df = df.rename(columns={'Single Race 6': 'Race'})

#Rename "Five-Year Age Groups" to "Age Group"
df = df.rename(columns={'Five-Year Age Groups': 'Age Group'})

#Rename "Education" to "Education Level"
df = df.rename(columns={'Education': 'Education Level'})

#Rename "Census Region" to "Region"
df = df.rename(columns={'Census Region': 'Region'})

#Rename	"Deaths" to "Total Deaths"
df = df.rename(columns={'Deaths': 'Total Deaths'})

#Rename	"% of Total Deaths" to "Percent of Total Deaths"
df = df.rename(columns={'% of Total Deaths': 'Percent of Total Deaths'})

#view changes
df.head()


# **Statistical Analysis**

# **Deaths by Age Group**

The maternal death totals per race for each age group

In [ ]:
# Group data by Race and Age Group, then calculate min, max, and sum of Total Deaths
region_deaths = df[df['Race'].isin(['Black or African American', 'White'])].groupby(['Race', 'Age Group'])['Total Deaths'].agg(['sum', 'min', 'max'])

# Display the results
print(region_deaths)


There is an increase in maternal deaths from 25 - 39. A bar plot can show the visual breakdown of each age group per race.

In [ ]:
import pandas as pd

data = {
    'Race': ['Black or African American'] * 6 + ['White'] * 6,
    'Age Group': ['15-19', '20-24', '25-29', '30-34', '35-39', '40-44'] * 2,
    'Total Deaths': [23, 123, 205, 315, 243, 86, 12, 183, 436, 569, 512, 222]
}

df_totals = pd.DataFrame(data)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set style
sns.set(style="whitegrid")

# Create grouped bar chart
g = sns.catplot(
    data=df_totals,
    x='Age Group',
    y='Total Deaths',
    hue='Race',
    kind='bar',
    palette='bright',
    height=5,
    aspect=2
)

# Customize labels and layout
g.set_axis_labels("Age Group", "Total Maternal Deaths")
g.fig.suptitle('Maternal Deaths by Age Group and Race', y=1.05)
g.set_xticklabels(rotation=45)

# Move legend to the right of the plot
g._legend.set_bbox_to_anchor((1.05, 0.8))
g._legend.set_loc('upper left')

plt.tight_layout()
plt.show()


The **age groups** with the highest maternal death totals are **25-29, 30-34, and 35-39. 30-34** bears the **highest death total for both Black and white women.**

A chi-squared test was used to identify the association between maternal deaths for Black and white women across different age groups.

**Chi-Squared Test**
1. **Hypothesis:** There is an association between race and maternal deaths across age groups.

2. **Null Hypothesis:** There is no association between race and maternal deaths across age groups.

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

# Create a contingency table
table = pd.crosstab(df['Race'], df['Age Group'])

# Run Chi-square
chi2, p, dof, expected = chi2_contingency(table)
print(f"Chi-square: {chi2:.2f}, p-value: {p:.4f}")

**We fail to reject the null hypothesis. With a P-value of 0.9017, this indicates that there is no significant relationship between race and age group at the 5% significance level.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'df' is your DataFrame
age_group_stats = df.groupby('Age Group')['Total Deaths'].agg(['min', 'max', 'mean', 'median'])

# Create the distribution plot
plt.figure(figsize=(12, 6))
ax = sns.histplot(data=df, x='Age Group', weights='Total Deaths', kde=True, bins=len(df['Age Group'].unique()))
plt.title('Distribution of Deaths by Age Group for Black and White Women')
plt.xlabel('Age Group')
plt.ylabel('Total Deaths')


plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Data is left skewed with women ages 25 - 39 having the highest maternal death totals for Black and white women of all education levels and regions.

Additionally, given that the average age of first-time mothers in the U.S. is approximately 27 ([Source](https://www.pewresearch.org/short-reads/2023/05/09/facts-about-u-s-mothers/)), women who begin childbearing at this age may still be within the maternal risk zone when having subsequent children in their 30s. This prolonged exposure to reproductive health risks could contribute to increased mortality rates in the 25-39 age group.

The central tendency highlights the overall pattern of maternal deaths across age groups for both racial groups.

In [ ]:
# Calculate central tendency measures for deaths by age group
age_group_stats = df.groupby('Age Group')['Total Deaths'].agg(['mean', 'median', 'std'])
age_group_stats


1. Youngest group (**15–19**) has the lowest and most **consistent number of deaths.**

2. **30–34** age group has the **highest average and median**, suggesting it may be the most vulnerable age range for **maternal death**.

3. High standard deviations in middle age groups suggest uneven distribution of deaths, possibly influenced by other factors (e.g., health conditions, region, race).



# ***Deaths By Region***

A comparative analysis of Black and white women by region can show the similarities and differences between the two groups.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Ensure 'Total Deaths' is numeric
df['Total Deaths'] = pd.to_numeric(df['Total Deaths'], errors='coerce')

# Group the data by Region and Race, and sum Total Deaths
grouped = df[df['Race'].isin(['Black or African American', 'White'])].groupby(['Region', 'Race'])['Total Deaths'].sum().reset_index()

# Plot the grouped data
sns.barplot(data=grouped,
            x='Region', y='Total Deaths', hue='Race', errorbar=None)
plt.title('Maternal Deaths by Region and Race')
plt.show()


1. **Deaths for Black and white women** are **higher in the Southern region** of the U.S., and white maternal deaths outnumber Black maternal deaths in the Midwest, possibly due to the higher population of white women in the Midwest.

2. **Maternal death data for Black women in the Western region were not available**, likely due to low case counts or data suppression to maintain confidentiality and protect individual privacy.

In [ ]:
# heatmap of regions and  deaths for black women

# Assuming 'df' is your DataFrame as defined in the provided code.

import matplotlib.pyplot as plt
import seaborn as sns

# Filter data for Black women
black_women_data = df[df['Race'] == 'Black or African American']

# Create the heatmap
plt.figure(figsize=(10, 6))
heatmap_data = black_women_data.pivot_table(index='Region', values='Total Deaths', aggfunc='sum')
sns.heatmap(heatmap_data, annot=True, fmt='g', cmap='viridis')
plt.title('Maternal Deaths for Black Women by Region')
plt.ylabel('Region')
plt.show()


1. The heatmap indicates that Black women experience higher maternal death rates in the Southern region of the United States.

2. It's important to consider that the Southern U.S. is home to 56% of the Black or African American population, which may contribute to the elevated maternal death rates observed in this region. [Source](https://www.pewresearch.org/race-and-ethnicity/fact-sheet/facts-about-the-us-black-population/)

3. Additionally, cardiovascular issues, a leading cause of maternal deaths in the U.S., are often exacerbated by obesity. The South has the highest obesity rates among Black women, which could influence these maternal mortality rates further. [Source](https://www.cdc.gov/obesity/data-and-statistics/adult-obesity-prevalence-maps.html)

4. Maternal death data for Black women in the Western region is unavailable, likely due to low case counts or data suppression to protect individual privacy and maintain confidentiality. Only 10% of the Black or African American population lives in the West region of the U.S. [Source](https://www.pewresearch.org/race-and-ethnicity/fact-sheet/facts-about-the-us-black-population/)

In [ ]:
# prompt: heatmap of regions and  total deaths for white women

import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'df' is your DataFrame from the previous code

# Filter for white women
white_women_df = df[(df['Race'] == 'White')]

# Create the heatmap
plt.figure(figsize=(10, 6))  # Adjust figure size as needed
heatmap_data = white_women_df.pivot_table(index='Region', values='Total Deaths', aggfunc='sum')
sns.heatmap(heatmap_data, annot=True, fmt="d", cmap="YlGnBu") # Use fmt="d" for integer annotation
plt.title('Maternal Deaths for White Women by Region')
plt.ylabel('Region')
plt.show()


1. The heatmap shows that white women have higher maternal death rates in the South region of the United States.
2.  Additionally, cardiovascular issues, a leading cause of maternal deaths in the U.S., are often exacerbated by obesity. The South has the highest obesity rates among white women, which could influence these maternal mortality rates further. [Source](https://www.cdc.gov/obesity/data-and-statistics/adult-obesity-prevalence-maps.html)
3. There is a greater spread of white maternal deaths across regions that Black women.

A Fisher’s Exact Test will be conducted to examine the association between race and region, specifically comparing the South versus non-South regions to further explore maternal death rates among Black and White women in the South.

## **Fisher's Exact Test**
1. **Hypothesis:** Black women are more likely to die from maternal causes in the South compared to other regions, when compared to White women.

2. **Null Hypothesis:** There is no association between race and region in maternal death counts — the odds of death in the South vs. Non-South are the same for Black and White women.

In [ ]:
import pandas as pd
from scipy.stats import fisher_exact

# Step 1: Create a 2x2 contingency table
# Format: [[Black South, Black Non-South], [White South, White Non-South]]
table = [[764, 231],  # Black
         [1030, 904]]  # White

# Step 2: Run Fisher's Exact Test
odds_ratio, p_value = fisher_exact(table)

# Step 3: Print the result
print("Odds Ratio:", odds_ratio)
print("P-value:", p_value)

**We reject the null hypothesis: Black women had approximately 3 times higher odds of dying in the South compared to non-South regions, relative to White women. This result is statistically significant at the 5% level.**

**Region vs Total Deaths Across Both Races**

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

figsize = (12, 1.2 * len(df['Region'].unique()))
plt.figure(figsize=figsize)

sns.violinplot(
    data=df,
    x='Total Deaths',
    y='Region',
    inner='stick',
    palette='Dark2',
    hue='Region',  # Add hue to match palette
    legend=False
)

sns.despine(top=True, right=True, bottom=True, left=True)
plt.title('Distribution of Total Maternal Deaths by Region')
plt.show()


Maternal death totals for Black and white women in the South region far outweigh those of other regions,
 highlighting the stark difference between the four regions.

# **Deaths by Education Level**

An overview of the total maternal death counts per education level.

In [ ]:
#total deaths per education level for Black women and white women
region_deaths = df[df['Race'].isin(['Black or African American', 'White'])].groupby(['Race', 'Education Level'])['Total Deaths'].sum()
print(region_deaths)

A boxplot is used to show the distribution of deaths by education level for both Black and white women. The boxplot identifies outliers and where most of the deaths occur per education level.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.boxplot(y='Education Level', x='Total Deaths', hue='Race', data=df,
            palette='tab10', width=0.6)

plt.title('Distribution of Deaths by Education Level and Race')
plt.ylabel('Education Level')
plt.xlabel('Total Deaths')
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better readability
plt.grid(axis='y', linestyle='--', alpha=0.7)  # Light horizontal gridlines
plt.tight_layout()
plt.show()


1. **High school graduate or GED completed has the most maternal deaths for both Black and White women**. The deaths are highly variable in this group, and the numbers are **right-skewed**, meaning that while most of the data points are clustered on the lower end, there are a few high outliers that significantly impact the overall total.

2. **Black women with some college credit, but not a degree**, have a relatively low number of maternal deaths in most cases. However, there are still some very high death counts in this group, creating a **right-skewed** distribution. This means that while most values are lower, a few extreme high values are pulling the mean upward.

3. **Right-skewed distributions** (like in the cases above) can be influenced by factors such as socioeconomic status, healthcare access, and other underlying health conditions, all of which can cause a small number of extreme cases that significantly affect the overall data.

4. Most **White women with some college credit, but not a degree**, tend to have relatively higher maternal death counts, but there are still cases with extremely low values, which create a **left-skewed** distribution. This indicates that most of the death counts are higher, but a few low values are pulling the data to the lower end.

A chi-squared test is used to identify the association between education level and race.

**Chi-Squared Test**

1.   **Hypothesis:** Black maternal deaths are higher when compared to white maternal deaths across education levels.
2.   **Null Hypothesis:** There is an association between race and maternal death counts across education levels.



In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

# Create a contingency table
table = pd.crosstab(df['Race'], df['Education Level'])

# Run Chi-square
chi2, p, dof, expected = chi2_contingency(table)
print(f"Chi-square: {chi2:.2f}, p-value: {p:.4f}")

**We fail to reject the null hypothesis. With a p-value of 0.7491, there is no statistically significant relationship between race and education level at the 5% significance level.**

The distribution of education level per race is further analyzed by exploring central tendency.

In [ ]:
# Filter only Black and White women
race_filtered = df[df['Race'].isin(['Black or African American', 'White'])]

# Group by both Race and Education Level
education_stats_by_race = race_filtered.groupby(['Race', 'Education Level'])['Total Deaths'].agg(['mean', 'median', 'std'])
education_stats_by_race

1. The **High school graduate or GED** group has a noticeably **high average** and extremely high standard deviation, possibly due to a larger population size or socioeconomic factors.

2. Higher education (**Master’s**) correlates with **lower** and more stable **death** **rates**, potentially due to better health literacy, access, and resources.

3. Women without completed degrees **(some college or no diploma) also face notable risk**, showing that education level may impact maternal health outcomes.

In [ ]:
# @title Education Level vs Total Deaths

from matplotlib import pyplot as plt
import seaborn as sns

figsize = (12, 1.2 * len(df['Education Level'].unique()))
plt.figure(figsize=figsize)

sns.violinplot(
    data=df,
    x='Total Deaths',
    y='Education Level',
    inner='stick',
    palette='Dark2',
    hue='Education Level',  # Assign hue to fix the warning
    legend=False            # Boolean, not string
)

sns.despine(top=True, right=True, bottom=True, left=True)
plt.title('Distribution of Total Maternal Deaths by Education Level')
plt.show()


There is a stark disparity in maternal mortality based on education level, with women holding only a **high school diploma or GED experiencing the highest death counts.** This pattern persists across racial groups, as both Black and white women with a high school education face the greatest mortality risks, followed by those with some college credit but no degree.

# **Comparison with all variables**

**Race vs Total Deaths**

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

figsize = (12, 1.2 * len(df['Race'].unique()))
plt.figure(figsize=figsize)

sns.violinplot(
    data=df,
    x='Total Deaths',
    y='Race',
    inner='stick',
    palette='Dark2',
    hue='Race',
    legend=False
)

sns.despine(top=True, right=True, bottom=True, left=True)
plt.title('Distribution of Total Maternal Deaths by Race')
plt.show()


Although the total number of Black women is significantly lower than that of White women in the overall population, the violin plot shows that the distribution of maternal deaths for Black women is close in size to that of White women.

This indicates that Black women experience maternal deaths at a disproportionately higher rate compared to White women.
In other words, despite representing a smaller portion of the population, Black women face a greater risk of maternal mortality, highlighting a critical racial disparity in health outcomes.



In [ ]:
# Step 1: Rename columns to remove spaces
df = df.rename(columns={
    'Total Deaths': 'Total_Deaths',
    'Education Level': 'Education_Level',
    'Age Group': 'Age_Group'
})

# Step 2: Fit the Poisson regression model
import statsmodels.api as sm
import statsmodels.formula.api as smf

poisson_model = smf.glm(
    formula="Total_Deaths ~ Race + Education_Level + Age_Group + Region",
    data=df,
    family=sm.families.Poisson()
).fit()

# Step 3: Display summary
print(poisson_model.summary())


This Poisson regression model evaluated total maternal deaths in the U.S. from 2018 to 2023 by race, education level, age group, and region, controlling for these variables simultaneously.

**Observation:**

1.   High school graduates or GED recipients had significantly higher death counts compared to those with less than high school (Coef = 1.16, p < 0.001).

2. Master’s degree holders had significantly lower deaths (Coef = –0.64, p < 0.001).

3. Risk increased steadily with age. Peak risk was for ages 30–34 (Coef = 1.42, p < 0.001), followed by 35–39 and 25–29.

4. South had the highest increase in maternal deaths (Coef = 0.87, p < 0.001).





# **Conclusion**

**The analysis of maternal death rates among Black and White women revealed the following key patterns:**

1. Women with a high school diploma or GED had the highest total death counts.

2. The age groups with the highest total deaths were 25-29, 30-34, and 35-39 years old.

3. Women in the South experienced a significantly higher number of maternal deaths compared to other regions.

Additionally, the analysis showed that **Black women in the South were three times more likely to die from pregnancy-related complications than White women**.
Despite being a smaller share of the overall population, Black women accounted for a disproportionately high number of maternal deaths, highlighting a significant racial disparity.


**Based on the findings, the general at-risk profile for maternal death is**:

1. Women aged 25-39

2. With a high school diploma or GED

3. Living in the South

**The highest-risk profile is:**

1. Black women aged 30-34

2. With a high school diploma or GED

3. Residing in the South

# **Future Scope**

TruBridge can use the identified risk profile to design targeted intervention strategies for pregnant women. By focusing on the groups most at risk, TruBridge can more effectively allocate resources and support services. Further analysis can also be conducted to examine underlying cofactors, such as obesity, access to healthcare, and other social determinants, that may contribute to the high maternal death rates within this demographic. Additionally, this tracking system can be integrated into family planning initiatives to help reduce maternal mortality among high-risk women.